# 🏠 BƯỚC 3: HUẤN LUYỆN MÔ HÌNH AI
**Dự án:** Dự đoán Giá Bất Động Sản Việt Nam  
**Phụ trách:** Thái (ML Engineer)  
**Mô tả:** Train 4 thuật toán ML, so sánh và xuất model tốt nhất

## 1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('✅ Import thư viện thành công!')

## 2. Load và kết hợp dữ liệu

In [ ]:
# Load 2 bộ dữ liệu
URL_CHUNG_CU = 'https://raw.githubusercontent.com/tangoctai2004/House-Price-Prediction/refs/heads/main/data/processed/cleaned_chung_cu.csv'
URL_NHA_DAT  = 'https://raw.githubusercontent.com/tangoctai2004/House-Price-Prediction/refs/heads/main/data/processed/cleaned_nha_dat.csv'

df_cc = pd.read_csv(URL_CHUNG_CU)
df_nd = pd.read_csv(URL_NHA_DAT)

df_cc['loai_bds'] = 'chung_cu'
df_nd['loai_bds'] = 'nha_dat'

print(f'📦 Chung cư: {df_cc.shape[0]} bản ghi, {df_cc.shape[1]} cột')
print(f'📦 Nhà đất : {df_nd.shape[0]} bản ghi, {df_nd.shape[1]} cột')

## 3. Tiền xử lý và tạo feature chung

In [ ]:
# ----- Chung cư -----
# Các cột chung: price_billion, area_m2, bedrooms_num, district, direction, furniture_std, legal_std
cc_cols = ['price_billion','area_m2','bedrooms_num','district','direction','furniture_std','legal_std','loai_bds']

# Nhà đất có thêm: floors_num, frontage_m, road_width_m
# Ta dùng tập đặc trưng chung để 2 loại cùng train 1 model tổng hợp
# Các cột riêng của nhà đất sẽ điền 0 vào chung cư
df_cc2 = df_cc.copy()
df_cc2['floors_num']   = 0
df_cc2['frontage_m']   = 0
df_cc2['road_width_m'] = 0

# Cột balcony_direction trong chung cư không có trong nhà đất -> bỏ
if 'balcony_direction' in df_cc2.columns:
    df_cc2 = df_cc2.drop(columns=['balcony_direction'])

ALL_COLS = ['price_billion','area_m2','bedrooms_num','district','direction',
            'furniture_std','legal_std','floors_num','frontage_m','road_width_m','loai_bds']

df_all = pd.concat([df_cc2[ALL_COLS], df_nd[ALL_COLS]], ignore_index=True)
print(f'\n📊 Dataset tổng hợp: {df_all.shape[0]} bản ghi')
print(f'Phân bổ loại BĐS:')
print(df_all['loai_bds'].value_counts())

In [ ]:
# ----- Làm sạch thêm -----
df_all = df_all.dropna(subset=['price_billion','area_m2'])

# Loại outlier giá: giữ 1 - 200 tỷ
df_all = df_all[(df_all['price_billion'] >= 1) & (df_all['price_billion'] <= 200)]

# Loại outlier diện tích: giữ 10 - 1000 m2
df_all = df_all[(df_all['area_m2'] >= 10) & (df_all['area_m2'] <= 1000)]

print(f'✅ Sau khi lọc outlier: {df_all.shape[0]} bản ghi')
print(df_all.describe())

## 4. Encoding biến phân loại

In [ ]:
df_model = df_all.copy()

cat_cols = ['district','direction','furniture_std','legal_std','loai_bds']

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

print('✅ Encoding hoàn tất!')
print('\nCác nhãn district:', list(label_encoders['district'].classes_))
print('Các nhãn furniture:', list(label_encoders['furniture_std'].classes_))
print('Các nhãn legal:', list(label_encoders['legal_std'].classes_))

## 5. Chia dữ liệu Train / Test (80:20)

In [ ]:
FEATURES = ['area_m2','bedrooms_num','district','direction','furniture_std','legal_std',
            'floors_num','frontage_m','road_width_m','loai_bds']
TARGET   = 'price_billion'

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'🎯 Train: {X_train.shape[0]} mẫu')
print(f'🎯 Test : {X_test.shape[0]} mẫu')
print(f'📐 Số features: {len(FEATURES)}')

In [ ]:
# Chuẩn hóa cho Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Chuẩn hóa dữ liệu xong!')

## 6. Hàm đánh giá mô hình

In [ ]:
results = {}

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'model': model, 'y_pred': y_pred}
    
    print(f'\n{'='*50}')
    print(f'🤖 {name}')
    print(f'  RMSE : {rmse:.4f} tỷ')
    print(f'  MAE  : {mae:.4f} tỷ')
    print(f'  R²   : {r2:.4f} ({r2*100:.1f}%)')
    return model

## 7. Train 4 thuật toán ML

In [ ]:
# ---- 1. Linear Regression (dùng data đã scale) ----
print('🔵 [1/4] Linear Regression...')
lr = evaluate_model('Linear Regression',
                    LinearRegression(),
                    X_train_scaled, X_test_scaled, y_train, y_test)

In [ ]:
# ---- 2. Decision Tree ----
print('🟡 [2/4] Decision Tree...')
dt = evaluate_model('Decision Tree',
                    DecisionTreeRegressor(max_depth=10, min_samples_split=5, random_state=42),
                    X_train, X_test, y_train, y_test)

In [ ]:
# ---- 3. Random Forest ----
print('🟢 [3/4] Random Forest...')
rf = evaluate_model('Random Forest',
                    RandomForestRegressor(n_estimators=200, max_depth=15,
                                         min_samples_split=4, random_state=42, n_jobs=-1),
                    X_train, X_test, y_train, y_test)

In [ ]:
# ---- 4. XGBoost ----
print('🔴 [4/4] XGBoost...')
xgb = evaluate_model('XGBoost',
                     XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=7,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=42, n_jobs=-1, verbosity=0),
                     X_train, X_test, y_train, y_test)

## 8. So sánh kết quả 4 mô hình

In [ ]:
compare_df = pd.DataFrame({
    name: {'RMSE (tỷ)': v['RMSE'], 'MAE (tỷ)': v['MAE'], 'R² Score': v['R2']}
    for name, v in results.items()
}).T.round(4)

print('\n' + '='*65)
print('📊 BẢNG SO SÁNH KẾT QUẢ 4 MÔ HÌNH')
print('='*65)
print(compare_df.to_string())

best_name = max(results, key=lambda x: results[x]['R2'])
print(f'\n🏆 MODEL TỐT NHẤT: {best_name}  (R² = {results[best_name]["R2"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
names = list(results.keys())
colors = ['#3498db','#f39c12','#2ecc71','#e74c3c']

for ax, metric, title, best_fn in zip(
    axes,
    ['RMSE','MAE','R2'],
    ['RMSE (thấp hơn tốt hơn)','MAE (thấp hơn tốt hơn)','R² Score (cao hơn tốt hơn)'],
    [min, min, max]
):
    vals = [results[n][metric] for n in names]
    best_val = best_fn(vals)
    bar_colors = ['gold' if v == best_val else c for v, c in zip(vals, colors)]
    bars = ax.bar(names, vals, color=bar_colors, edgecolor='black', linewidth=0.7)
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(names, rotation=15, ha='right')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('So sánh 4 Thuật toán ML - Dự đoán Giá BĐS', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu biểu đồ so sánh!')

## 9. Xuất model tốt nhất (.pkl)

In [ ]:
os.makedirs('../models', exist_ok=True)

best_model = results[best_name]['model']

# Lưu best model
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Lưu scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Lưu label encoders
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Lưu metadata
model_meta = {
    'best_model_name': best_name,
    'features': FEATURES,
    'target': TARGET,
    'metrics': {k: v for k, v in results[best_name].items() if k in ['RMSE','MAE','R2']},
    'all_results': {n: {k: v for k, v in d.items() if k in ['RMSE','MAE','R2']} for n, d in results.items()}
}
with open('../models/model_meta.pkl', 'wb') as f:
    pickle.dump(model_meta, f)

print('✅ Đã xuất các file:')
print('   📁 models/best_model.pkl      — Não AI tốt nhất')
print('   📁 models/scaler.pkl          — Bộ chuẩn hóa')
print('   📁 models/label_encoders.pkl  — Bộ mã hóa nhãn')
print('   📁 models/model_meta.pkl      — Metadata mô hình')
print(f'\n🏆 Model xuất: {best_name}')

## 10. Feature Importance (cho Random Forest & XGBoost)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_name, model_obj in [
    (axes[0], 'Random Forest', results['Random Forest']['model']),
    (axes[1], 'XGBoost',       results['XGBoost']['model'])
]:
    importances = model_obj.feature_importances_
    fi_df = pd.DataFrame({'feature': FEATURES, 'importance': importances})\
              .sort_values('importance', ascending=True)
    
    colors_fi = ['#e74c3c' if i == fi_df['importance'].idxmax() else '#3498db'
                 for i in fi_df.index]
    ax.barh(fi_df['feature'], fi_df['importance'], color=colors_fi)
    ax.set_title(f'Feature Importance — {model_name}', fontweight='bold')
    ax.set_xlabel('Importance')
    for i, (_, row) in enumerate(fi_df.iterrows()):
        ax.text(row['importance'] + 0.001, i, f'{row["importance"]:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../models/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu biểu đồ Feature Importance!')

## 11. Test dự đoán nhanh

In [ ]:
def predict_price(area_m2, bedrooms, district_name, furniture, legal, loai='chung_cu',
                  floors=0, frontage=0, road_width=0, direction='Không rõ'):
    """
    Dự đoán giá BĐS theo tỷ VNĐ.
    """
    sample = pd.DataFrame([{
        'area_m2': area_m2, 'bedrooms_num': bedrooms,
        'district': district_name, 'direction': direction,
        'furniture_std': furniture, 'legal_std': legal,
        'floors_num': floors, 'frontage_m': frontage,
        'road_width_m': road_width, 'loai_bds': loai
    }])
    for col in cat_cols:
        le = label_encoders[col]
        val = sample[col].astype(str).values[0]
        if val in le.classes_:
            sample[col] = le.transform([val])[0]
        else:
            sample[col] = 0  # fallback nếu giá trị chưa từng thấy
    return best_model.predict(sample[FEATURES])[0]

# --- Thử nghiệm ---
test_cases = [
    ('Chung cư 75m², 2PN, Cầu Giấy, Đầy đủ, Sổ đỏ',
     dict(area_m2=75, bedrooms=2, district_name='Cầu Giấy',
          furniture='Đầy đủ', legal='Sổ đỏ/Sổ hồng', loai='chung_cu')),
    ('Chung cư 50m², 2PN, Hà Đông, Cơ bản, HĐMB',
     dict(area_m2=50, bedrooms=2, district_name='Hà Đông',
          furniture='Cơ bản', legal='HĐMB', loai='chung_cu')),
    ('Nhà đất 45m², 4PN, Thanh Xuân, Đầy đủ, Sổ đỏ, 5 tầng',
     dict(area_m2=45, bedrooms=4, district_name='Thanh Xuân',
          furniture='Đầy đủ', legal='Sổ đỏ/Sổ hồng', loai='nha_dat',
          floors=5, frontage=4.2, road_width=4.0)),
]

print('🔮 THỬ NGHIỆM DỰ ĐOÁN GIÁ:')
print('='*60)
for desc, params in test_cases:
    price = predict_price(**params)
    print(f'  📍 {desc}')
    print(f'     → Giá dự đoán: {price:.2f} tỷ VNĐ (~{price*1000:.0f} triệu)\n')

## ✅ Tổng kết

| File xuất | Mô tả |
|---|---|
| `models/best_model.pkl` | Não AI tốt nhất |
| `models/scaler.pkl` | Bộ chuẩn hóa dữ liệu |
| `models/label_encoders.pkl` | Bộ mã hóa nhãn |
| `models/model_meta.pkl` | Metadata & kết quả so sánh |
| `models/model_comparison.png` | Biểu đồ so sánh |
| `models/feature_importance.png` | Feature Importance |

**Xem chi tiết đánh giá tại: `04_evaluation.ipynb`**